# Data Preparation Pipeline

Loads EMS call data and CalEnviroScreen hazard data, spatially joins to census tracts, and produces call metrics by 3-month periods (mean & std of calls/week per tract).

In [1]:
import clean

## Full pipeline (one call)

In [2]:
df = clean.prepare_analysis_data(
    ems_path='Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260315.csv',
    hazards_path='calenviroscreen40resultsdatadictionary_F_2021.xlsx',
)

In [3]:
print(f"Rows (census tracts): {df.shape[0]}, Columns: {df.shape[1]}")

Rows (census tracts): 182, Columns: 66


In [4]:
call_cols = [c for c in df.columns if 'call_' in c]
print(f"Call metric columns: {call_cols}")

Call metric columns: ['call_avg_aug_oct', 'call_avg_feb_apr', 'call_avg_may_jul', 'call_avg_nov_jan', 'call_std_aug_oct', 'call_std_feb_apr', 'call_std_may_jul', 'call_std_nov_jan']


In [5]:
df[clean.HAZARD_COVARIATES + call_cols].describe()

,ozone,pm2.5,diesel_pm,drinking_water,lead,pesticides,tox._release,traffic,cleanup_sites,groundwater_threats,...,unemployment,housing_burden,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,...,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,0.031054,8.623882,0.726596,260.157563,58.828028,0.009934,350.977366,1040.861682,8.222527,9.593956,...,4.191209,15.592308,23.041264,23.032527,22.286538,22.995110,8.229560,9.084286,7.395824,8.654231
std,0.001300,0.066518,0.694990,10.785832,14.029507,0.034081,50.752971,627.605542,15.244609,19.090027,...,2.514737,6.628387,29.914483,30.029962,29.148713,29.251723,6.649992,8.577673,6.012284,7.434762
min,0.029372,8.445628,0.079785,258.789440,6.432362,0.000000,221.798665,158.686585,0.000000,0.000000,...,0.000000,4.500000,3.220000,2.890000,3.420000,2.430000,1.480000,1.600000,1.620000,0.980000
25%,0.029372,8.585158,0.240545,258.789440,53.493155,0.000000,314.711669,561.393627,0.000000,0.300000,...,2.600000,11.225000,9.230000,9.112500,8.930000,9.440000,4.645000,4.612500,4.292500,4.902500
50%,0.031908,8.616646,0.475722,258.789440,60.074576,0.000000,350.686831,804.165108,1.750000,3.750000,...,3.750000,14.300000,13.750000,13.860000,13.180000,14.285000,6.250000,6.705000,5.655000,6.480000
75%,0.031908,8.660773,0.989824,258.789440,66.666802,0.000000,387.638250,1415.432537,9.000000,13.225000,...,5.400000,18.550000,20.927500,20.395000,19.735000,21.105000,8.620000,9.182500,7.935000,8.897500
max,0.034190,8.797361,4.751602,363.855387,91.056201,0.272686,499.944661,3273.393341,113.400000,151.800000,...,18.100000,39.000000,236.570000,231.140000,232.500000,237.640000,46.250000,62.780000,48.950000,57.670000


In [6]:
df[['census_tract'] + call_cols + clean.HAZARD_COVARIATES].head()

,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan,ozone,...,imp._water_bodies,solid_waste,asthma,low_birth_weight,cardiovascular_disease,education,linguistic_isolation,poverty,unemployment,housing_burden
0,6075023200,27.86,25.86,25.79,27.86,7.99,10.20,9.97,13.85,0.031908,...,11,20.75,123.98,8.09,11.96,25.3,10.4,27.3,6.1,37.0
1,6075023103,18.21,13.86,17.29,17.00,8.29,7.03,8.77,7.42,0.030640,...,14,13.90,123.98,7.74,11.96,21.0,4.5,71.7,6.4,22.3
2,6075023001,13.64,9.92,12.50,15.77,5.60,4.07,6.12,6.51,0.031908,...,10,5.75,123.98,6.41,11.96,27.5,19.8,30.7,4.8,25.6
3,6075023400,15.43,20.64,20.64,16.57,7.08,8.04,8.94,6.35,0.031908,...,11,16.90,123.98,6.29,11.96,29.7,23.1,40.1,6.3,10.4
4,6075023102,22.08,22.64,19.57,22.07,8.27,11.08,6.55,8.12,0.031908,...,14,9.30,123.98,8.70,11.96,17.8,5.5,48.1,11.6,25.5


## Step-by-step breakdown

In [7]:
hazards, demographics, dictionary = clean.load_hazards_data(
    'calenviroscreen40resultsdatadictionary_F_2021.xlsx'
)
hazards_sf = clean.filter_san_francisco(hazards)
print(f"SF census tracts in hazards data: {len(hazards_sf)}")

SF census tracts in hazards data: 195


In [8]:
ems = clean.load_ems_data(
    'Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260315.csv'
)
med_inc = clean.filter_medical_incidents(ems)
print(f"Total EMS calls: {len(ems)}, Medical incidents: {len(med_inc)}")

Total EMS calls: 368900, Medical incidents: 241317


In [9]:
med_inc = clean.extract_coords(med_inc, location_col='case_location')
med_inc = clean.assign_nearest_tract(med_inc, hazards_sf)
med_inc = clean.assign_period(med_inc)
call_metrics = clean.aggregate_calls_by_period(med_inc)
print(f"Tracts with call metrics: {len(call_metrics)}")
call_metrics.head()

Tracts with call metrics: 194


,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan
0,6075010100,18.00,16.07,19.79,14.07,8.62,5.95,8.55,8.89
1,6075010200,17.00,13.14,17.86,20.79,5.96,6.49,6.31,7.53
2,6075010300,22.29,24.00,23.86,23.50,8.30,9.13,7.15,7.62
3,6075010400,10.71,14.57,13.29,14.50,4.94,8.03,6.39,3.74
4,6075010500,41.00,45.21,41.00,36.43,13.99,17.03,8.03,10.75


In [11]:
sf_hazards = clean.merge_hazards_and_calls(hazards_sf, call_metrics)
print(f"Final analysis shape: {result.shape}")
call_cols = [c for c in result.columns if 'call_' in c]
sf_hazards[['census_tract'] + call_cols + clean.HAZARD_COVARIATES].head()

Final analysis shape: (182, 66)


,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan,ozone,...,imp._water_bodies,solid_waste,asthma,low_birth_weight,cardiovascular_disease,education,linguistic_isolation,poverty,unemployment,housing_burden
0,6075023200,27.86,25.86,25.79,27.86,7.99,10.20,9.97,13.85,0.031908,...,11,20.75,123.98,8.09,11.96,25.3,10.4,27.3,6.1,37.0
1,6075023103,18.21,13.86,17.29,17.00,8.29,7.03,8.77,7.42,0.030640,...,14,13.90,123.98,7.74,11.96,21.0,4.5,71.7,6.4,22.3
2,6075023001,13.64,9.92,12.50,15.77,5.60,4.07,6.12,6.51,0.031908,...,10,5.75,123.98,6.41,11.96,27.5,19.8,30.7,4.8,25.6
3,6075023400,15.43,20.64,20.64,16.57,7.08,8.04,8.94,6.35,0.031908,...,11,16.90,123.98,6.29,11.96,29.7,23.1,40.1,6.3,10.4
4,6075023102,22.08,22.64,19.57,22.07,8.27,11.08,6.55,8.12,0.031908,...,14,9.30,123.98,8.70,11.96,17.8,5.5,48.1,11.6,25.5
